# Finetune SEMamba (RoyChao19477/SEMamba) trên dữ liệu Noise-Speech

Notebook này:
1. Clone repo SEMamba và cài đặt môi trường (torch, mamba-ssm, ...).
2. Đọc dữ liệu từ 3 thư mục bạn cung cấp trên Kaggle:
   - `.../NOISE SPEECH/TRAIN/CLEAN`
   - `.../NOISE SPEECH/TRAIN/NOISE`
   - `.../NOISE SPEECH/TEST ` (chú ý có dấu cách sau `TEST`)
3. Sinh các file `data/*.json` mà `train.py` cần (dataloader của repo này match cặp clean/noisy **theo tên file giống hệt nhau**, nên notebook sẽ tự kiểm tra và cảnh báo nếu tên file không khớp).
4. Tạo checkpoint khởi đầu từ trọng số pretrained (`ckpts/SEMamba_advanced.pth` + `ckpts/pretrained_discriminator.pth`) để `train.py` **resume/finetune** thay vì train from scratch.
5. Chạy finetune bằng `train.py` gốc của repo (không sửa source).
6. (Tuỳ chọn) Chạy inference/enhance trên tập TEST.

> ⚠️ Lưu ý: SEMamba bắt buộc chạy trên GPU (không hỗ trợ CPU). Hãy bật GPU (Settings → Accelerator → GPU) trước khi chạy.


## 0. Kiểm tra GPU

In [1]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print("CUDA available:", torch.cuda.is_available())
print("Num GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available(), "Không thấy GPU! Vào Settings > Accelerator, bật GPU rồi chạy lại."


Tue Aug  4 07:19:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone repo SEMamba

In [2]:
%cd /kaggle/working
!rm -rf SEMamba
!git clone --depth 1 https://github.com/RoyChao19477/SEMamba.git
%cd /kaggle/working/SEMamba
!ls


/kaggle/working
Cloning into 'SEMamba'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 203 (delta 19), reused 170 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 25.13 MiB | 32.96 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/kaggle/working/SEMamba
ckpts		 imgs		  mamba-1_2_0_post1  README.md	       run.sh
data		 inference.py	  mamba_install      recipes	       train.py
dataloaders	 LICENSE	  models	     requirements.txt  utils
EnhancedSamples  make_dataset.sh  pretrained.sh      runPCS.sh


### 1b. Fix xung đột tên package (`utils`, `models`, `dataloaders`)

Các thư mục `utils/`, `models/`, `dataloaders/` trong repo **không có file `__init__.py`**, nên Python coi chúng là "namespace package" — ưu tiên thấp hơn bất kỳ package pip nào trùng tên (ví dụ Kaggle/pip có sẵn 1 package tên `utils`). Hệ quả là `train.py` có thể báo `ModuleNotFoundError: No module named 'utils.util'` dù file vẫn tồn tại. Cell dưới tạo `__init__.py` rỗng để các thư mục này luôn được ưu tiên đúng.


In [3]:
import os

for pkg_dir in ["utils", "models", "dataloaders"]:
    init_file = os.path.join(pkg_dir, "__init__.py")
    if not os.path.exists(init_file):
        open(init_file, "w").close()
        print(f"Da tao {init_file}")
    else:
        print(f"{init_file} da ton tai")


Da tao utils/__init__.py
Da tao models/__init__.py
Da tao dataloaders/__init__.py


## 2. Cài đặt thư viện

Kaggle GPU image đã có sẵn `torch` (thường tương thích tốt), nên mặc định **không** ép cài lại `torch==2.2.2` để tránh xung đột CUDA driver. Nếu sau này gặp lỗi liên quan tới torch/mamba-ssm, hãy mở cell bị comment bên dưới ra chạy thử.


In [4]:
!pip install -q packaging librosa soundfile pyyaml tensorboard pesq einops joblib triton ninja

# Nếu Kaggle báo lỗi phiên bản torch không tương thích với repo, bỏ comment 2 dòng dưới
# (kiểm tra CUDA của Kaggle bằng `!nvidia-smi` rồi chọn build cu12x phù hợp):
# !pip install -q torch==2.2.2 torchaudio==2.2.2


  Preparing metadata (setup.py) ... done


In [5]:
# Kiem tra nvcc (bat buoc phai co de build mamba-ssm tu source neu wheel PyPI khong khop)
import subprocess
r = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(r.stdout, r.stderr)
print("nvcc co san:" , r.returncode == 0)


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
 
nvcc co san: True


In [6]:
# CACH THAY THE neu build tu source that bai: thu de setup.py cua mamba-ssm
# tu tai wheel dung san tu GitHub Releases (khop dung torch/cuda/python cua Kaggle),
# thay vi ep build local (thuong fail vi Kaggle khong co san nvcc).
import subprocess, torch

print("torch:", torch.__version__, "| cuda:", torch.version.cuda)

def run_show(cmd, cwd=None, env=None):
    print("$", " ".join(cmd))
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
    print(p.stdout[-4000:])
    if p.returncode != 0:
        print(p.stderr[-4000:])
    print("Return code:", p.returncode)
    return p.returncode

# Khong ghim version cu (1.2.0.post1) nua - de pip/setup.py tu chon wheel prebuilt
# khop nhat voi torch/cuda/python hien tai cua Kaggle.
rc = run_show(["pip", "install", "mamba-ssm", "causal-conv1d", "--no-build-isolation"])

if rc == 0:
    import importlib
    import mamba_ssm
    importlib.reload(mamba_ssm)
    print("mamba_ssm da san sang (wheel prebuilt tu GitHub Releases).")
else:
    print("Van khong duoc. Copy toan bo log stderr o tren gui lai de debug tiep "
          "(tim dong co chu 'error:', 'nvcc', 'CUDA_HOME', hoac 'No matching distribution').")


torch: 2.10.0+cu128 | cuda: 12.8
$ pip install mamba-ssm causal-conv1d --no-build-isolation
ocal/lib/python3.12/dist-packages (from rich>=12.3.0->typer->huggingface-hub<2.0,>=1.3.0->transformers->mamba-ssm) (4.0.0)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.7/670.7 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.6/770.6 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 17.9 MB/s eta 0

In [7]:
import importlib
import mamba_ssm, torch, einops, librosa, soundfile, pesq
importlib.reload(mamba_ssm)
print("mamba_ssm OK, torch", torch.__version__, "| CUDA build:", torch.version.cuda)
print("mamba_ssm version:", getattr(mamba_ssm, '__version__', 'khong ro'))


mamba_ssm OK, torch 2.10.0+cu128 | CUDA build: 12.8
mamba_ssm version: 2.3.2.post1


### 2b. Patch tương thích `mamba_ssm` mới (bắt buộc)

Code gốc của SEMamba (2024) viết cho `mamba-ssm <= 1.2.x`, trong đó `Block` nằm trong `mamba_ssm.modules.mamba_simple`. Các bản `mamba-ssm >= 2.x` (thứ mà bước cài đặt ở trên có thể đã tự tải về vì có sẵn wheel cho Python 3.12) đã **dời `Block` sang `mamba_ssm.modules.block`** và bắt buộc phải truyền thêm tham số `mlp_cls`. Cell dưới ghi đè lại `models/mamba_block.py` trong bản clone để tự nhận diện và tương thích cả 2 phiên bản — không đổi logic mô hình.


In [8]:
mamba_block_patched = r'''# Reference: https://github.com/state-spaces/mamba/blob/9127d1f47f367f5c9cc49c73ad73557089d02cb8/mamba_ssm/models/mixer_seq_simple.py
# NOTE: da patch de tuong thich ca mamba_ssm <=1.2.x va >=2.x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import init
from torch.nn.parameter import Parameter
from functools import partial
from einops import rearrange

from mamba_ssm.modules.mamba_simple import Mamba

try:
    # mamba_ssm <= 1.2.x: Block nam trong mamba_simple, khong can mlp_cls
    from mamba_ssm.modules.mamba_simple import Block
    _NEEDS_MLP_CLS = False
except ImportError:
    # mamba_ssm >= 2.x: Block chuyen sang modules.block, bat buoc truyen mlp_cls
    from mamba_ssm.modules.block import Block
    _NEEDS_MLP_CLS = True

from mamba_ssm.models.mixer_seq_simple import _init_weights

try:
    from mamba_ssm.ops.triton.layernorm import RMSNorm
except ImportError:
    from mamba_ssm.ops.triton.layer_norm import RMSNorm

def create_block(
    d_model, cfg, layer_idx=0, rms_norm=True, fused_add_norm=False, residual_in_fp32=False,
    ):
    d_state = cfg['model_cfg']['d_state']
    d_conv = cfg['model_cfg']['d_conv']
    expand = cfg['model_cfg']['expand']
    norm_epsilon = cfg['model_cfg']['norm_epsilon']

    mixer_cls = partial(Mamba, layer_idx=layer_idx, d_state=d_state, d_conv=d_conv, expand=expand)
    norm_cls = partial(
        nn.LayerNorm if not rms_norm else RMSNorm, eps=norm_epsilon
    )
    block_kwargs = {'mlp_cls': nn.Identity} if _NEEDS_MLP_CLS else {}
    block = Block(
            d_model,
            mixer_cls,
            norm_cls=norm_cls,
            fused_add_norm=fused_add_norm,
            residual_in_fp32=residual_in_fp32,
            **block_kwargs,
            )
    block.layer_idx = layer_idx
    return block

class MambaBlock(nn.Module):
    def __init__(self, in_channels, cfg):
        super(MambaBlock, self).__init__()
        n_layer = 1
        self.forward_blocks  = nn.ModuleList( create_block(in_channels, cfg) for i in range(n_layer) )
        self.backward_blocks = nn.ModuleList( create_block(in_channels, cfg) for i in range(n_layer) )

        self.apply(
            partial(
                _init_weights,
                n_layer=n_layer,
            )
        )

    def forward(self, x):
        x_forward, x_backward = x.clone(), torch.flip(x, [1])
        resi_forward, resi_backward = None, None

        for layer in self.forward_blocks:
            x_forward, resi_forward = layer(x_forward, resi_forward)
        y_forward = (x_forward + resi_forward) if resi_forward is not None else x_forward

        for layer in self.backward_blocks:
            x_backward, resi_backward = layer(x_backward, resi_backward)
        y_backward = torch.flip((x_backward + resi_backward), [1]) if resi_backward is not None else torch.flip(x_backward, [1])

        return torch.cat([y_forward, y_backward], -1)

class TFMambaBlock(nn.Module):
    # Temporal-Frequency Mamba block for sequence modeling.
    def __init__(self, cfg):
        super(TFMambaBlock, self).__init__()
        self.cfg = cfg
        self.hid_feature = cfg['model_cfg']['hid_feature']

        self.time_mamba = MambaBlock(in_channels=self.hid_feature, cfg=cfg)
        self.freq_mamba = MambaBlock(in_channels=self.hid_feature, cfg=cfg)

        self.tlinear = nn.ConvTranspose1d(self.hid_feature * 2, self.hid_feature, 1, stride=1)
        self.flinear = nn.ConvTranspose1d(self.hid_feature * 2, self.hid_feature, 1, stride=1)

    def forward(self, x):
        b, c, t, f = x.size()

        x = x.permute(0, 3, 2, 1).contiguous().view(b*f, t, c)
        x = self.tlinear( self.time_mamba(x).permute(0,2,1) ).permute(0,2,1) + x
        x = x.view(b, f, t, c).permute(0, 2, 1, 3).contiguous().view(b*t, f, c)
        x = self.flinear( self.freq_mamba(x).permute(0,2,1) ).permute(0,2,1) + x
        x = x.view(b, t, f, c).permute(0, 3, 1, 2)
        return x
'''

with open("models/mamba_block.py", "w") as f:
    f.write(mamba_block_patched)

print("Da ghi de models/mamba_block.py voi ban tuong thich ca 2 API cua mamba_ssm.")


Da ghi de models/mamba_block.py voi ban tuong thich ca 2 API cua mamba_ssm.


In [9]:
# Kiem tra patch hoat dong: import thu model generator (dung toan bo pipeline import
# giong luc train.py chay that). Neu loi se hien ro o day, truoc khi chay finetune that su.
import importlib, sys

for mod in list(sys.modules):
    if mod.startswith("models"):
        del sys.modules[mod]

from models.generator import SEMamba
from models.discriminator import MetricDiscriminator
print("Import models.generator / models.discriminator THANH CONG. Patch hoat dong dung.")


Import models.generator / models.discriminator THANH CONG. Patch hoat dong dung.


### 2c. Patch `batch_pesq` trong `models/discriminator.py` (bat buoc)

Ham `batch_pesq` dung `cfg['env_setting']['num_workers']` lam `n_jobs` cho `joblib.Parallel`.
Nhung o Buoc 7 ta da dat `num_workers = 0` de tranh deadlock CUDA+fork cho `DataLoader`
(2 muc dich khac nhau dang dung chung 1 tham so). `joblib` khong chap nhan `n_jobs=0`
(`ValueError: n_jobs == 0 in Parallel has no meaning`). Cell duoi tach rieng: ep
`batch_pesq` luon chay `n_jobs=1` (chay tuan tu, khong fork them tien trinh), khong
phu thuoc vao `cfg` nua.


In [10]:
disc_path = "models/discriminator.py"
with open(disc_path, "r") as f:
    disc_src = f.read()

old = "def batch_pesq(clean, noisy, cfg):\n    num_worker = cfg['env_setting']['num_workers']\n    pesq_score = Parallel(n_jobs=num_worker)(delayed(pesq_loss)(c, n) for c, n in zip(clean, noisy))"
new = "def batch_pesq(clean, noisy, cfg):\n    # SUA: khong dung cfg['env_setting']['num_workers'] nua (gia tri nay = 0 de\n    # tranh deadlock DataLoader, nhung joblib khong chap nhan n_jobs=0).\n    # Chay tuan tu (n_jobs=1) de an toan, khong fork them tien trinh.\n    pesq_score = Parallel(n_jobs=1)(delayed(pesq_loss)(c, n) for c, n in zip(clean, noisy))"

assert old in disc_src, "Khong tim thay doan can patch, kiem tra lai file goc."
disc_src = disc_src.replace(old, new)

with open(disc_path, "w") as f:
    f.write(disc_src)

print("Da patch batch_pesq trong models/discriminator.py de dung n_jobs=1 co dinh.")


Da patch batch_pesq trong models/discriminator.py de dung n_jobs=1 co dinh.


## 3. Khai báo đường dẫn dữ liệu & kiểm tra tồn tại

In [11]:
import os, glob

TRAIN_CLEAN_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
TRAIN_NOISY_DIR = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"
TEST_DIR        = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST "  # co dau cach sau "TEST"

paths = {"TRAIN_CLEAN_DIR": TRAIN_CLEAN_DIR, "TRAIN_NOISY_DIR": TRAIN_NOISY_DIR, "TEST_DIR": TEST_DIR}
missing = []
for name, p in paths.items():
    ok = os.path.isdir(p)
    print(f"{name}: {'OK' if ok else 'KHONG TIM THAY'}  ->  {p!r}")
    if not ok:
        missing.append(name)

if missing:
    print("\nMot so duong dan khong ton tai. Dang tim tu dong trong /kaggle/input ...")
    candidates = glob.glob("/kaggle/input/**/NOISE SPEECH", recursive=True)
    print("Cac thu muc 'NOISE SPEECH' tim thay:")
    for c in candidates:
        print(" -", repr(c))
    print("\n=> Neu thay duong dan dung o tren, hay copy va gan lai vao TRAIN_CLEAN_DIR / TRAIN_NOISY_DIR / TEST_DIR roi chay lai cell nay.")


TRAIN_CLEAN_DIR: OK  ->  '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN'
TRAIN_NOISY_DIR: OK  ->  '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE'
TEST_DIR: OK  ->  '/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST '


## 4. Khảo sát nhanh cấu trúc dữ liệu

In [12]:
def list_wavs(directory):
    files = []
    for root, dirs, filenames in os.walk(directory):
        for fn in filenames:
            if fn.lower().endswith('.wav'):
                files.append(os.path.join(root, fn))
    return files

for name, path in [("TRAIN/CLEAN", TRAIN_CLEAN_DIR), ("TRAIN/NOISE", TRAIN_NOISY_DIR), ("TEST", TEST_DIR)]:
    print(f"--- {name} ({path}) ---")
    if not os.path.isdir(path):
        print("  (khong ton tai, bo qua)\n")
        continue
    sub_entries = os.listdir(path)
    print("  Entries cap 1 (toi da 10):", sub_entries[:10])
    wavs = list_wavs(path)
    print("  Tong so file .wav (bao gom thu muc con):", len(wavs))
    print("  Vi du ten file:", [os.path.basename(w) for w in wavs[:5]])
    print()


--- TRAIN/CLEAN (/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN) ---
  Entries cap 1 (toi da 10): ['AINAV017-N28.wav', 'RLQNV015-N01.wav', 'RLQNV017-N14.wav', 'RLBDV015-N09.wav', 'RLMNV017-N33.wav', 'RLMNV017-N35.wav', 'RLQNV009-N15.wav', 'AINAV019-N23.wav', 'AIMBV019-N05.wav', 'AINAV004-N39.wav']
  Tong so file .wav (bao gom thu muc con): 8460
  Vi du ten file: ['AINAV017-N28.wav', 'RLQNV015-N01.wav', 'RLQNV017-N14.wav', 'RLBDV015-N09.wav', 'RLMNV017-N33.wav']

--- TRAIN/NOISE (/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE) ---
  Entries cap 1 (toi da 10): ['AINAV017-N28.wav', 'RLQNV015-N01.wav', 'RLQNV017-N14.wav', 'RLBDV015-N09.wav', 'RLMNV017-N33.wav', 'RLMNV017-N35.wav', 'RLQNV009-N15.wav', 'AINAV019-N23.wav', 'AIMBV019-N05.wav', 'AINAV004-N39.wav']
  Tong so file .wav (bao gom thu muc con): 8460
  Vi du ten file: ['AINAV017-N28.wav', 'RLQNV015-N01.wav', 'RLQNV017-N14.wav', 'RLBDV015-N09.wav', 'RLMNV017-N33.wav']

--- TEST (/kag

## 5. Tạo `data/train_*.json` và `data/valid_*.json`

`dataloaders/dataloader_vctk.py` của repo ghép cặp clean/noisy **theo tên file (basename) giống hệt nhau**. Vì vậy bước dưới sẽ:
- Liệt kê toàn bộ `.wav` trong `TRAIN/CLEAN` và `TRAIN/NOISE`.
- Chỉ giữ lại các file có tên trùng nhau ở cả hai bên (và cảnh báo nếu có file bị lệch).
- Trích một phần nhỏ làm tập validation (dùng để theo dõi PESQ trong lúc finetune).


In [13]:
import json, random

clean_files = list_wavs(TRAIN_CLEAN_DIR)
noisy_files = list_wavs(TRAIN_NOISY_DIR)
print(f"CLEAN: {len(clean_files)} file | NOISE: {len(noisy_files)} file")

clean_by_name = {os.path.basename(f): f for f in clean_files}
noisy_by_name = {os.path.basename(f): f for f in noisy_files}

common = sorted(set(clean_by_name) & set(noisy_by_name))
only_noisy = sorted(set(noisy_by_name) - set(clean_by_name))
only_clean = sorted(set(clean_by_name) - set(noisy_by_name))

print(f"So cap ten file KHOP giua CLEAN va NOISE: {len(common)}")
if only_noisy:
    print(f"CANH BAO: {len(only_noisy)} file trong NOISE khong co ban CLEAN cung ten. VD: {only_noisy[:5]}")
if only_clean:
    print(f"CANH BAO: {len(only_clean)} file trong CLEAN khong co ban NOISE cung ten. VD: {only_clean[:5]}")

assert len(common) > 0, (
    "Khong tim thay cap file CLEAN/NOISE nao trung ten! "
    "Kiem tra lai: 2 thu muc phai chua file .wav CUNG TEN cho cung 1 cau noi (1 ban sach, 1 ban co nhieu)."
)


CLEAN: 8460 file | NOISE: 8460 file
So cap ten file KHOP giua CLEAN va NOISE: 8460


In [14]:
random.seed(1234)
common_shuffled = common[:]
random.shuffle(common_shuffled)

VALID_RATIO = 0.05  # 5% du lieu train lam validation, co the chinh lai
n_valid = max(1, int(len(common_shuffled) * VALID_RATIO))
valid_names = common_shuffled[:n_valid]
train_names = common_shuffled[n_valid:]

print(f"Train: {len(train_names)} cap | Valid: {len(valid_names)} cap")

def dump_json(name_to_path, names, out_path):
    paths_out = [name_to_path[n] for n in names]
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'w') as f:
        json.dump(paths_out, f, indent=4)
    print(f"Da ghi {len(paths_out)} duong dan vao {out_path}")

dump_json(clean_by_name, train_names, "data/train_clean.json")
dump_json(noisy_by_name, train_names, "data/train_noisy.json")
dump_json(clean_by_name, valid_names, "data/valid_clean.json")
dump_json(noisy_by_name, valid_names, "data/valid_noisy.json")


Train: 8037 cap | Valid: 423 cap
Da ghi 8037 duong dan vao data/train_clean.json
Da ghi 8037 duong dan vao data/train_noisy.json
Da ghi 423 duong dan vao data/valid_clean.json
Da ghi 423 duong dan vao data/valid_noisy.json


## 6. Xử lý tập TEST

Vì chưa biết cấu trúc bên trong `TEST` (có tách CLEAN/NOISE hay chỉ là các file lẫn nhiễu để enhance), cell dưới sẽ tự phát hiện:
- Nếu `TEST` có thư mục con dạng `CLEAN` và `NOISE`/`NOISY` → build `data/test_clean.json` / `data/test_noisy.json` giống cách làm ở Bước 5 (dùng để đánh giá PESQ khách quan).
- Nếu không → dùng tập validation ở trên làm `test_*.json` (để `train.py` không lỗi khi đọc config), còn toàn bộ file trong `TEST` sẽ được dùng để chạy **inference/enhance** trực tiếp ở phần cuối notebook.


In [15]:
import shutil

test_wavs = list_wavs(TEST_DIR)
print(f"So file .wav trong TEST: {len(test_wavs)}")

subdirs = [d for d in os.listdir(TEST_DIR) if os.path.isdir(os.path.join(TEST_DIR, d))] if os.path.isdir(TEST_DIR) else []
print("Thu muc con trong TEST:", subdirs)

clean_sub, noisy_sub = None, None
for d in subdirs:
    dl = d.lower()
    if 'clean' in dl and clean_sub is None:
        clean_sub = os.path.join(TEST_DIR, d)
    if ('noise' in dl or 'noisy' in dl) and noisy_sub is None:
        noisy_sub = os.path.join(TEST_DIR, d)

if clean_sub and noisy_sub:
    print(f"Phat hien cau truc TEST/CLEAN ({clean_sub}) + TEST/NOISE ({noisy_sub})")
    tc_by_name = {os.path.basename(f): f for f in list_wavs(clean_sub)}
    tn_by_name = {os.path.basename(f): f for f in list_wavs(noisy_sub)}
    common_test = sorted(set(tc_by_name) & set(tn_by_name))
    print(f"So cap test khop ten: {len(common_test)}")
    if common_test:
        dump_json(tc_by_name, common_test, "data/test_clean.json")
        dump_json(tn_by_name, common_test, "data/test_noisy.json")
    else:
        print("Khong co cap nao khop ten trong TEST, fallback sang dung valid set.")
        shutil.copy("data/valid_clean.json", "data/test_clean.json")
        shutil.copy("data/valid_noisy.json", "data/test_noisy.json")
else:
    print("TEST khong co cau truc CLEAN/NOISE ro rang.")
    print("-> Dung tap valid o Buoc 5 lam test_clean.json/test_noisy.json (de train.py doc config khong loi).")
    shutil.copy("data/valid_clean.json", "data/test_clean.json")
    shutil.copy("data/valid_noisy.json", "data/test_noisy.json")
    print(f"-> {len(test_wavs)} file trong TEST se duoc dung de chay inference/enhance truc tiep o Buoc 9.")


So file .wav trong TEST: 940
Thu muc con trong TEST: []
TEST khong co cau truc CLEAN/NOISE ro rang.
-> Dung tap valid o Buoc 5 lam test_clean.json/test_noisy.json (de train.py doc config khong loi).
-> 940 file trong TEST se duoc dung de chay inference/enhance truc tiep o Buoc 9.


## 7. Tạo file config cho finetune

In [16]:
import yaml, torch

with open("recipes/SEMamba_advanced/SEMamba_advanced.yaml") as f:
    cfg = yaml.safe_load(f)

num_gpus = max(1, torch.cuda.device_count())

cfg['env_setting']['num_gpus'] = num_gpus
# SUA: num_workers>0 + CUDA context da tao truoc (generator.to(device) trong train.py)
# gay deadlock do fork() tien trinh con sau khi CUDA da khoi tao (rat hay gap tren Kaggle).
# Trieu chung dung y: dung im sau dong 'Epoch: 1', khong loi, khong in step nao.
cfg['env_setting']['num_workers'] = 0           # tranh deadlock CUDA+fork DataLoader
cfg['env_setting']['checkpoint_interval'] = 200
cfg['env_setting']['validation_interval'] = 200
cfg['env_setting']['summary_interval'] = 20
cfg['env_setting']['stdout_interval'] = 10
cfg['env_setting']['dist_cfg']['dist_url'] = 'tcp://localhost:19478'

cfg['training_cfg']['training_epochs'] = 20     # finetune it epoch hon train tu dau, co the chinh
# SUA: batch_size=4*num_gpus se OOM vi Cell chay train.py ben duoi chi
# cho thay 1 GPU (CUDA_VISIBLE_DEVICES="0"), nen tat ca batch se don het
# vao 1 GPU T4 15GB va tran bo nho ngay trong epoch dau (xem Cell 9-uoc luong).
# Dat batch_size co dinh, khong nhan theo num_gpus:
cfg['training_cfg']['batch_size'] = 4
cfg['training_cfg']['learning_rate'] = 0.00005  # LR nho hon ~10 lan so voi train tu dau (0.0005) de finetune on dinh
cfg['training_cfg']['use_PCS400'] = False

EXP_NAME = "SEMamba_finetune"
os.makedirs("recipes/SEMamba_finetune", exist_ok=True)
CONFIG_PATH = "recipes/SEMamba_finetune/finetune.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"So GPU phat hien: {num_gpus}")
print(yaml.dump(cfg, sort_keys=False))


So GPU phat hien: 2
env_setting:
  num_gpus: 2
  num_workers: 0
  seed: 1234
  stdout_interval: 10
  checkpoint_interval: 200
  validation_interval: 200
  summary_interval: 20
  dist_cfg:
    dist_backend: nccl
    dist_url: tcp://localhost:19478
    world_size: 1
data_cfg:
  train_clean_json: data/train_clean.json
  train_noisy_json: data/train_noisy.json
  valid_clean_json: data/valid_clean.json
  valid_noisy_json: data/valid_noisy.json
  test_clean_json: data/test_clean.json
  test_noisy_json: data/test_noisy.json
training_cfg:
  training_epochs: 20
  batch_size: 4
  learning_rate: 5.0e-05
  adam_b1: 0.8
  adam_b2: 0.99
  lr_decay: 0.99
  segment_size: 32000
  loss:
    metric: 0.05
    magnitude: 0.9
    phase: 0.3
    complex: 0.1
    time: 0.2
    consistancy: 0.1
  use_PCS400: false
stft_cfg:
  sampling_rate: 16000
  n_fft: 400
  hop_size: 100
  win_size: 400
model_cfg:
  hid_feature: 64
  compress_factor: 0.3
  num_tfmamba: 4
  d_state: 16
  d_conv: 4
  expand: 4
  norm_epsilon

## 8. Khởi tạo checkpoint để finetune từ trọng số pretrained

`train.py` gốc tự resume nếu tìm thấy `g_XXXXXXXX.pth` và `do_XXXXXXXX.pth` trong `exp/<exp_name>/`. Ta tận dụng đúng cơ chế đó: nạp sẵn trọng số pretrained (`ckpts/SEMamba_advanced.pth` cho generator, `ckpts/pretrained_discriminator.pth` cho discriminator) rồi lưu ra đúng định dạng, thay vì sửa `train.py`.


In [17]:
import torch
from models.generator import SEMamba
from models.discriminator import MetricDiscriminator
from utils.util import load_config

cfg = load_config(CONFIG_PATH)
EXP_PATH = f"exp/{EXP_NAME}"
os.makedirs(EXP_PATH, exist_ok=True)

device = torch.device("cuda:0")
generator = SEMamba(cfg).to(device)
discriminator = MetricDiscriminator().to(device)

# Nap trong so pretrained cho generator
g_ckpt = torch.load("ckpts/SEMamba_advanced.pth", map_location=device)
missing, unexpected = generator.load_state_dict(g_ckpt['generator'], strict=False)
print("Da nap generator pretrained. missing keys:", len(missing), "| unexpected keys:", len(unexpected))

# Nap trong so pretrained cho discriminator (neu tuong thich)
try:
    d_state = torch.load("ckpts/pretrained_discriminator.pth", map_location=device)
    discriminator.load_state_dict(d_state, strict=False)
    print("Da nap discriminator pretrained tu ckpts/pretrained_discriminator.pth")
except Exception as e:
    print("Khong nap duoc discriminator pretrained (se train discriminator tu dau):", e)

# Optimizer moi khoi tao, chi de dung dinh dang cho co che resume cua train.py
lr = cfg['training_cfg']['learning_rate']
betas = (cfg['training_cfg']['adam_b1'], cfg['training_cfg']['adam_b2'])
optim_g = torch.optim.AdamW(generator.parameters(), lr=lr, betas=betas)
optim_d = torch.optim.AdamW(discriminator.parameters(), lr=lr, betas=betas)

torch.save({'generator': generator.state_dict()}, f"{EXP_PATH}/g_00000000.pth")
torch.save({
    'discriminator': discriminator.state_dict(),
    'optim_g': optim_g.state_dict(),
    'optim_d': optim_d.state_dict(),
    'steps': 0,
    'epoch': -1,
}, f"{EXP_PATH}/do_00000000.pth")

print("Da tao checkpoint khoi dau cho finetune tai:", EXP_PATH)
del generator, discriminator, optim_g, optim_d
torch.cuda.empty_cache()


Da nap generator pretrained. missing keys: 0 | unexpected keys: 0
Da nap discriminator pretrained tu ckpts/pretrained_discriminator.pth
Da tao checkpoint khoi dau cho finetune tai: exp/SEMamba_finetune


## 9. Chạy finetune

`train.py` sẽ tự phát hiện `g_00000000.pth` / `do_00000000.pth` trong `exp/SEMamba_finetune/` và tiếp tục train (finetune) từ đó thay vì train from scratch.

> Có thể mất khá lâu tuỳ số lượng file và số epoch. Theo dõi log PESQ ở Validation để biết khi nào nên dừng sớm (Kaggle notebook có giới hạn thời gian chạy).


In [ ]:
import subprocess, os

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
# SUA: giam phan manh bo nho GPU, giup tranh OOM khi train Mamba tren T4
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["PYTHONPATH"] = "/kaggle/working/SEMamba" + os.pathsep + env.get("PYTHONPATH", "")

cmd = [
    "python", "train.py",
    "--config", CONFIG_PATH,
    "--exp_folder", "exp",
    "--exp_name", EXP_NAME,
]
print("Chay:", " ".join(cmd))

proc = subprocess.Popen(cmd, env=env, cwd="/kaggle/working/SEMamba",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nKet thuc voi return code:", proc.returncode)


Chay: python train.py --config recipes/SEMamba_finetune/finetune.yaml --exp_folder exp --exp_name SEMamba_finetune
/kaggle/working/SEMamba/train.py:343: UserWarning: Warning: The actual number of available GPUs (1) is less than the .yaml config (2). Auto reset to num_gpu = 1
  warnings.warn(
Number of GPUs available: 1
GPU 0: Tesla T4
Batch size per GPU: 4
SEMamba(
  (dense_encoder): DenseEncoder(
    (dense_conv_1): Sequential(
      (0): Conv2d(2, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): PReLU(num_parameters=64)
    )
    (dense_block): DenseBlock(
      (dense_block): ModuleList(
        (0): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
          (2): PReLU(num_parameters=64)
        )
        (1): Sequential(
          (0): Conv2d(128,

## 10. (Tuỳ chọn) Theo dõi bằng TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /kaggle/working/SEMamba/exp/SEMamba_finetune/logs


## 11. Chạy inference/enhance trên tập TEST

Chọn checkpoint generator mới nhất vừa finetune xong (`g_XXXXXXXX.pth` có số step lớn nhất trong `exp/SEMamba_finetune/`).

Nếu `TEST` có cấu trúc CLEAN/NOISE riêng (đã tạo `data/test_noisy.json` ở Bước 6), có thể dùng luôn `data/test_noisy.json`. Nếu không, cell dưới sẽ chạy enhance trực tiếp lên toàn bộ file `.wav` phẳng trong `TEST`.


In [ ]:
import glob

ckpts = sorted(glob.glob(f"{EXP_PATH}/g_*.pth"))
assert ckpts, "Chua co checkpoint nao duoc luu (chua du 1 checkpoint_interval step). Kiem tra lai buoc train o tren."
latest_ckpt = ckpts[-1]
print("Se dung checkpoint:", latest_ckpt)


In [ ]:
import subprocess, os

OUTPUT_DIR = "/kaggle/working/enhanced_test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["PYTHONPATH"] = "/kaggle/working/SEMamba" + os.pathsep + env.get("PYTHONPATH", "")

cmd = [
    "python", "inference.py",
    "--input_folder", TEST_DIR,      # inference.py doc truc tiep tat ca file trong thu muc nay (khong de quy)
    "--output_folder", OUTPUT_DIR,
    "--checkpoint_file", latest_ckpt,
    "--config", CONFIG_PATH,
    "--post_processing_PCS", "False",
]
print("Chay:", cmd)

proc = subprocess.Popen(cmd, env=env, cwd="/kaggle/working/SEMamba",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nKet thuc voi return code:", proc.returncode)
print("File enhance da luu tai:", OUTPUT_DIR)


### Ghi chú

- Nếu `TEST` có chứa thư mục con (không phải file `.wav` phẳng), `inference.py` gốc (`os.listdir` không đệ quy) sẽ không đọc được — khi đó hãy trỏ `--input_folder` thẳng vào thư mục con chứa file `.wav`, hoặc chạy lặp qua từng thư mục con.
- Muốn train lâu hơn/ngắn hơn: chỉnh `training_epochs`, `checkpoint_interval` ở Bước 7 rồi chạy lại từ Bước 8 (tạo lại checkpoint khởi đầu) — hoặc chạy lại Bước 9 nhiều lần, `train.py` sẽ tự resume từ checkpoint mới nhất trong `exp/SEMamba_finetune/`.
- Tăng `learning_rate` nếu thấy PESQ validation không cải thiện, giảm nếu thấy loss dao động mạnh/NaN.
